<a href="https://colab.research.google.com/github/HeenaBathyal/RAKSHAKsih/blob/main/Croprecommend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install keras_tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 2.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import matplotlib.pyplot as plt
import math

# RGB to Color Name Mapping
def rgb_to_color_name(r, g, b):
    colors = {
        'Healthy Green': (50, 205, 50),
        'Yellowish Green': (154, 205, 50),
        'Light Green': (144, 238, 144),
        'Dark Green': (0, 100, 0),
        'Disease Yellow': (255, 255, 0),
        'Blight Brown': (165, 42, 42)
    }

    r, g, b = int(r), int(g), int(b)
    min_distance = float('inf')
    closest_color = "Unknown"

    for name, (cr, cg, cb) in colors.items():
        distance = math.sqrt((r - cr)**2 + (g - cg)**2 + (b - cb)**2)
        if distance < min_distance:
            min_distance = distance
            closest_color = name

    return closest_color

# Load and preprocess data
print("Loading and preprocessing data...")
crop_recommend_df = pd.read_csv('/content/croprecommendpracticum.csv')
crop_analysis_df = pd.read_csv('/content/crop_analysis_results (4).csv')

# Clean data
crop_recommend_df = crop_recommend_df.drop_duplicates()
label_encoder = LabelEncoder()
crop_recommend_df['label'] = label_encoder.fit_transform(crop_recommend_df['label'])

# Prepare features and target
X = crop_recommend_df.drop('label', axis=1)
feature_names = X.columns.tolist()
y = crop_recommend_df['label']

# Define valid input ranges
feature_ranges = [
    (0, 150), (0, 150), (0, 150),  # N, P, K
    (0, 50), (0, 100),             # Temp, Humidity
    (0, 14), (0, 500)              # pH, Rainfall
]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
num_crops = len(label_encoder.classes_)

# Enhanced NCF Model with Dropout
def create_ncf_model(num_features, num_crops):
    input_layer = layers.Input(shape=(num_features,))
    x = layers.Dense(128, activation='relu')(input_layer)
    x = layers.Dropout(0.3)(x)  # Monte Carlo dropout

    crop_input = layers.Input(shape=(1,))
    y = layers.Embedding(num_crops, 64)(crop_input)
    y = layers.Flatten()(y)

    merged = layers.Concatenate()([x, y])
    merged = layers.Dense(64, activation='relu')(merged)
    merged = layers.Dropout(0.3)(merged)  # Additional dropout
    output = layers.Dense(1, activation='sigmoid')(merged)

    return keras.Model(inputs=[input_layer, crop_input], outputs=output)

# Prepare NCF data
def prepare_ncf_data(X, y, num_crops):
    X_pos = X
    y_pos = y.values.reshape(-1, 1)
    pos_labels = np.ones_like(y_pos)

    X_neg = np.repeat(X, 2, axis=0)  # Reduced negatives
    y_neg = np.random.randint(0, num_crops, size=(len(X)*2, 1))
    neg_labels = np.zeros_like(y_neg)

    X_combined = np.vstack([X_pos, X_neg])
    y_combined = np.vstack([y_pos, y_neg])
    labels = np.vstack([pos_labels, neg_labels])

    indices = np.arange(len(labels))
    np.random.shuffle(indices)
    return X_combined[indices], y_combined[indices], labels[indices]

# Train model
print("Preparing NCF data...")
X_ncf, y_ncf, labels_ncf = prepare_ncf_data(X_scaled, y, num_crops)

# Split NCF data into training and testing sets
X_train_ncf, X_test_ncf, y_train_ncf, y_test_ncf, labels_train, labels_test = train_test_split(
    X_ncf, y_ncf, labels_ncf, test_size=0.2, random_state=42
)

print("Training model...")
model = create_ncf_model(X.shape[1], num_crops)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(
    [X_train_ncf, y_train_ncf], labels_train,
    batch_size=64,
    epochs=100,
    validation_data=([X_test_ncf, y_test_ncf], labels_test),
    verbose=1
)

# Process crop traits
def process_crop_analysis_data(df):
    df = df[df['Crop Label'].str.endswith('.jpg', na=False)]
    df['Crop Label'] = df['Crop Label'].str.replace('.jpg', '').str.capitalize()

    numeric_cols = ['Leaf Color R', 'Leaf Color G', 'Leaf Color B',
                   'Leaf Height (px)', 'Plant Height (px)']

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna(subset=numeric_cols)
    return df.groupby('Crop Label')[numeric_cols].mean()

avg_traits = process_crop_analysis_data(crop_analysis_df)

# Input validation
def validate_input(input_values):
    valid = True
    for i, (value, (min_val, max_val)) in enumerate(zip(input_values, feature_ranges)):
        if not (min_val <= value <= max_val):
            print(f"Warning: {feature_names[i]}={value} is outside trained range ({min_val}-{max_val})")
            valid = False
    return valid

# Prediction with uncertainty estimation
def predict_with_uncertainty(model, input_features, scaler, n_samples=100):
    input_scaled = scaler.transform([input_features])
    all_crops = np.arange(num_crops).reshape(-1, 1)
    repeated_input = np.repeat(input_scaled, num_crops, axis=0)

    predictions = []
    for _ in range(n_samples):
        pred = model.predict([repeated_input, all_crops], verbose=0).flatten()
        predictions.append(pred)

    predictions = np.array(predictions)
    mean_conf = predictions.mean(axis=0)
    std_conf = predictions.std(axis=0)

    top_idx = np.argmax(mean_conf)
    return (
        label_encoder.inverse_transform([top_idx])[0],
        mean_conf[top_idx],
        std_conf[top_idx]
    )

# Confidence scaling
def scale_confidence(raw_confidence):
    # Map 0.5→0.7, 1.0→1.0
    return min(0.7 + 0.3*(raw_confidence - 0.5)/0.5, 1.0)

# Main recommendation function
def recommend_crop(input_features):
    if not validate_input(input_features):
        print("Note: Recommendation may be unreliable due to out-of-range inputs\n")

    crop, confidence, uncertainty = predict_with_uncertainty(model, input_features, scaler)
    scaled_confidence = scale_confidence(confidence)

    print("\nRecommended Crop:")
    if uncertainty > 0.15:
        print("⚠️ Low-confidence warning ⚠️")

    print(f"- {crop.capitalize()} (confidence: {scaled_confidence:.2%})")
    if uncertainty > 0:
        print(f"- Uncertainty: ±{uncertainty:.2%}")

    traits = avg_traits.loc[crop.capitalize()] if crop.capitalize() in avg_traits.index else None
    if traits is not None:
        color_name = rgb_to_color_name(traits['Leaf Color R'], traits['Leaf Color G'], traits['Leaf Color B'])
        print("\nPhenotypic Traits:")
        print(f"- Leaf Color: {color_name}")
        print(f"- Leaf Height: {traits['Leaf Height (px)']:.1f} px")
        print(f"- Plant Height: {traits['Plant Height (px)']:.1f} px")
    else:
        print("\nNo phenotypic data available for this crop")

# User input
def get_user_input():
    print("\nEnter the environmental conditions:")
    inputs = []
    for i, feature in enumerate(feature_names):
        while True:
            try:
                value = float(input(f"{feature} ({feature_ranges[i][0]}-{feature_ranges[i][1]}): "))
                if feature_ranges[i][0] <= value <= feature_ranges[i][1]:
                    inputs.append(value)
                    break
                print(f"Value must be between {feature_ranges[i][0]} and {feature_ranges[i][1]}")
            except ValueError:
                print("Please enter a valid number")
    return inputs

# Main program
print("\nAgricultural Crop Recommendation System")
print("--------------------------------------")

while True:
    print("\nOptions:")
    print("1. Get recommendation")
    print("2. Exit")

    choice = input("Enter choice: ").strip()

    if choice == '1':
        try:
            user_input = get_user_input()
            recommend_crop(user_input)
        except Exception as e:
            print(f"Error: {str(e)}")
    elif choice == '2':
        print("Exiting program...")
        break
    else:
        print("Invalid choice")

Loading and preprocessing data...
Preparing NCF data...
Training model...
Epoch 1/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.4958 - loss: 0.7002 - val_accuracy: 0.6778 - val_loss: 0.6350
Epoch 2/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6419 - loss: 0.6554 - val_accuracy: 0.6778 - val_loss: 0.6316
Epoch 3/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6701 - loss: 0.6303 - val_accuracy: 0.6778 - val_loss: 0.6245
Epoch 4/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6339 - loss: 0.6409 - val_accuracy: 0.6778 - val_loss: 0.6100
Epoch 5/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6611 - loss: 0.6228 - val_accuracy: 0.6778 - val_loss: 0.5912
Epoch 6/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6736 - loss: 0.5902 - val_accuracy: 0.6778 - val_loss: 0.5672
Epoch 7/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.6864 - loss: 0.5643 - val_accuracy: 0.6944 - val_loss: 0.5425
Epoch 8/100
12/12 ━━━━━━━━━━━━

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



Recommended Crop:
- Rice (confidence: 68.83%)
- Uncertainty: ±0.00%

Phenotypic Traits:
- Leaf Color: Yellowish Green
- Leaf Height: 85.7 px
- Plant Height: 285.8 px

Options:
1. Get recommendation
2. Exit

Enter the environmental conditions:


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



Recommended Crop:
- Rice (confidence: 68.83%)
- Uncertainty: ±0.00%

Phenotypic Traits:
- Leaf Color: Yellowish Green
- Leaf Height: 85.7 px
- Plant Height: 285.8 px

Options:
1. Get recommendation
2. Exit
